# Can a Model Detect Its Own Failure?
## A benchmark of confidence, uncertainty, calibration, and error prediction

**Research question.** Can a classifier anticipate its mistakes using information available *before* the ground-truth outcome is known?

**Design.** Two tabular datasets, four classifiers, development-only selection, cross-fitted failure detectors, independent calibration, selective prediction, controlled input corruptions, and paired uncertainty estimates. All reported numbers are computed by running the notebook. A model that appears confident can still fail. None of the procedures guarantees safe decisions.

**Run on Kaggle:** CPU notebook, Python, internet **ON** for the public Adult Census download, or add `adult.data` and `adult.test` as Kaggle dataset inputs. Outputs are written to `/kaggle/working/` (or `./outputs/` outside Kaggle). `FAST_MODE` provides a quicker smoke test. Recommended kernel: Kaggle Python with scikit-learn, pandas, numpy, matplotlib, scipy, seaborn (optional).

## Google Colab edition: important setup

This notebook does **not** need Kaggle, a GPU, or an external download for its default Breast Cancer Wisconsin experiment. It runs with the Python libraries normally installed in Colab. The Adult Census Income experiment is optional and requires the public dataset, either provided as uploaded files or retrieved using OpenML.

**Browser warning:** A `TypeError: Failed to fetch` involving `chrome-extension://...` **at notebook opening time** happens before notebook Python executes. No change to notebook code can guarantee a fix for that browser or extension problem. Open Colab in Incognito with extensions disabled, or use a fresh browser profile. Test with a blank notebook first.

**Instructions:** In Colab select **File → Upload notebook**, choose this `.ipynb`, then **Runtime → Run all**. Leave `RUN_DATASETS=['cancer']` initially. Files are written to `/content/failure_benchmark_outputs` in Colab. Execute the optional final zip cell to download generated CSV outputs.

For Adult data, upload `adult.data` and `adult.test` into `/content/` (or set `ADULT_DATA_DIR` to their containing directory), then add `'adult'` to `RUN_DATASETS`. If the files are absent, the notebook tries OpenML, which requires working external network access.

## 1. Title, abstract, and research motivation
A classifier's predicted class answers *what?*; a failure detector answers *is this prediction likely wrong?* Calibration evaluates probability agreement; error detection assesses the ranking of *errors*. Selective prediction trades automatic coverage for lower accepted-case risk. These are distinct quantities.

**Practical aim:** refer an explicitly budgeted proportion of cases to a human while reporting both the missed errors and correctly predicted cases unnecessarily referred.

## 2. Research questions and hypotheses
H1: Errors tend to have lower maximum confidence than correct predictions (not universally). H2: In binary classification, entropy and inverse margin have exactly the same rank ordering as 1−maximum probability, apart from ties. H3: Calibration can improve probabilistic quality without improving error ranking. H4: Cross-fitted learned risk can outperform confidence under some settings but not necessarily all. H5: Abstaining on high-risk cases reduces retained error rate only if error ranking is informative. H6: detection performance may deteriorate under shifts.

H1, H3–H6 are tested as empirical comparisons, not presumed findings. H2 is a mathematical prediction.

## 3. Theory and definitions
For probabilities $p_1,\ldots,p_K$ and hard prediction $\hat y$:

- **Confidence risk:** $1-\max_k p_k$.
- **Normalized predictive entropy:** $-\sum_k p_k\log(p_k)/\log K$.
- **Inverse top-two margin:** $1-(p_{(1)}-p_{(2)})$.
- **Ensemble mutual-information proxy:** $H(\bar p)-M^{-1}\sum_m H(p_m)$, nonnegative up to numerical error. This measures disagreement, not guaranteed epistemic uncertainty.
- **Failure label:** $e=\mathbb{1}[\hat y\ne y]$. Labels are used *only* to train or score detectors after each earlier prediction was generated independently.
- **Selective risk:** errors among accepted predictions. **Coverage:** fraction accepted.

For binary $p=(q,1-q)$, all three single-model scores are monotone functions of $\min(q,1-q)$. Hence AUROC/AUPRC must match, apart from floating ties, and they are not independent signals. ROC-AUC for failure uses `error=1`, not the original classification label.

## 4. Environment setup and reproducibility
Deterministic seeds, all package versions logged, CPU-friendly defaults. Refit all preprocessing within the training fold. The held-out final test is never used to choose the base model, detector, or review threshold.

In [ ]:
import os, sys, json, time, warnings, platform, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn
from scipy.stats import rankdata
from sklearn.datasets import load_breast_cancer, fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
 f1_score, roc_auc_score, average_precision_score, log_loss, brier_score_loss,
 confusion_matrix, roc_curve, precision_recall_curve)
SEED=42
FAST_MODE=os.environ.get('FAST_MODE','0')=='1'
# Colab defaults to the offline cancer dataset. Set RUN_DATASETS='cancer,adult' after confirming OpenML access.
RUN_DATASETS=[s.strip() for s in os.environ.get('RUN_DATASETS','cancer').split(',') if s.strip()]
assert RUN_DATASETS and set(RUN_DATASETS) <= {'cancer', 'adult'}, 'Choose cancer and/or adult'
OUT=Path('/kaggle/working' if Path('/kaggle/working').exists() else '/content/failure_benchmark_outputs' if Path('/content').exists() else './failure_benchmark_outputs')
OUT.mkdir(parents=True,exist_ok=True)
np.random.seed(SEED)
plt.rcParams.update({'figure.figsize':(8,4.5),'axes.grid':True,'grid.alpha':.2,'font.size':10,'figure.dpi':110})
print('FAST_MODE:',FAST_MODE,'datasets:',RUN_DATASETS,'outputs:',OUT)
print('versions:',{'python':sys.version.split()[0],'numpy':np.__version__,'pandas':pd.__version__,'scikit-learn':sklearn.__version__,'matplotlib':plt.matplotlib.__version__ if hasattr(plt,'matplotlib') else 'installed'})


## 5. Dataset loading and description
**Breast Cancer Wisconsin Diagnostic:** 569 records, 30 numeric features extracted from digitized cell nuclei, UCI via scikit-learn. This is a historical benchmark and is not a clinical deployment validation. **Adult Census Income:** 1994 U.S. census tabular records, numerical and categorical attributes, prediction of annual income >$50K, UCI Adult. Historical social patterns, selection effects, protected attributes, and label limitations affect interpretation.

Source information: [WDBC UCI](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic), [Adult UCI](https://archive.ics.uci.edu/dataset/2/adult). UCI database terms and attribution should be checked for downstream redistribution. Dataset files are not embedded. For Adult, local Kaggle input `adult.data` / `adult.test` is preferred if internet is disabled; otherwise `fetch_openml(data_id=1590)` uses the OpenML mirror and requires internet. An unavailable Adult dataset raises a clear error rather than quietly substituting unrelated data.

In [ ]:
def get_dataset(name):
    if name=='cancer':
        d=load_breast_cancer(as_frame=True)
        return d.data.copy(),pd.Series(d.target,name='target').astype(int), 'WDBC / sklearn'
    if name=='adult':
        files=glob.glob(str(Path(os.environ.get('ADULT_DATA_DIR', '/content'))/'adult.data')) + glob.glob('/kaggle/input/**/adult.data',recursive=True)
        if files:
            columns=['age','workclass','fnlwgt','education','education-num','marital-status','occupation','relationship','race','sex','capital-gain','capital-loss','hours-per-week','native-country','income']
            frames=[]
            for f in [files[0],files[0].replace('adult.data','adult.test')]:
                if not Path(f).exists():continue
                z=pd.read_csv(f,names=columns,skipinitialspace=True,comment='|',na_values=['?'],skiprows=1 if f.endswith('adult.test') else 0)
                frames.append(z)
            df=pd.concat(frames,ignore_index=True)
            y=df.pop('income').astype(str).str.rstrip('.').str.strip().eq('>50K').astype(int)
            return df,y,'Adult UCI from local CSV input'
        try:
            d=fetch_openml(data_id=1590,as_frame=True,parser='auto')
            X=d.data.copy().replace('?',np.nan)
            y=d.target.astype(str).str.rstrip('.').str.strip().eq('>50K').astype(int)
            if y.nunique()!=2:raise ValueError('unexpected Adult label encoding')
            return X,y,'Adult via OpenML data_id=1590'
        except Exception as exc:
            raise RuntimeError('Adult dataset unavailable. Enable Kaggle Internet or attach UCI adult.data/adult.test input files. Original error: '+str(exc)) from exc
    raise ValueError(name)

def make_split(X,y):
    X0,Xtest,y0,ytest=train_test_split(X,y,test_size=.20,stratify=y,random_state=SEED)
    Xtrain,Xdev,ytrain,ydev=train_test_split(X0,y0,test_size=.25,stratify=y0,random_state=SEED+1)
    assert len(set(Xtrain.index)&set(Xdev.index))==0 and len(set(Xdev.index)&set(Xtest.index))==0
    return Xtrain,Xdev,Xtest,ytrain,ydev,ytest


## 6. Exploratory data analysis
Inspect class balance, missingness, numeric distributions, outliers, and numerical correlation without fitting on test data. EDA is descriptive and cannot be used to fine-tune choices based on final test outcomes.

In [ ]:
def eda(X,y,title):
    display(pd.DataFrame({'count':y.value_counts().sort_index(),'share':y.value_counts(normalize=True).sort_index().round(3)}))
    print('Shape:',X.shape,'Missing fraction:',round(float(X.isna().mean().mean()),4))
    display(X.head(3))
    numeric=X.select_dtypes(include='number').columns
    display(X.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_fraction'))
    if len(numeric):
        sample=X[numeric].iloc[:,:min(6,len(numeric))]
        display(sample.describe().round(2))
        print('IQR outlier counts (exploratory):',((sample.lt(sample.quantile(.25)-1.5*(sample.quantile(.75)-sample.quantile(.25)))) | (sample.gt(sample.quantile(.75)+1.5*(sample.quantile(.75)-sample.quantile(.25))))).sum().to_dict())
        fig,axes=plt.subplots(1,3,figsize=(13,3.5))
        y.value_counts().sort_index().plot.bar(ax=axes[0],title='Class frequencies')
        sample.iloc[:,:min(3,len(numeric))].hist(ax=axes[1]) if False else axes[1].hist(sample.iloc[:,0].dropna(),bins=25)
        axes[1].set_title('Numeric example: '+str(sample.columns[0]))
        c=sample.corr(numeric_only=True)
        im=axes[2].imshow(c,vmin=-1,vmax=1,cmap='coolwarm');axes[2].set_xticks(range(len(c)),c.columns,rotation=90,fontsize=7);axes[2].set_yticks(range(len(c)),c.index,fontsize=7)
        fig.colorbar(im,ax=axes[2],fraction=.045);plt.suptitle(title+' EDA');plt.tight_layout();plt.show()


## 7. Data preprocessing and leakage-safe splitting
Partition: 60% base-development pool, 20% independent development (base selection, calibration, risk-threshold selection), and 20% untouched test. Inside the 60% training pool, 3-fold out-of-fold predictions create meta-training labels. Calibrators are fitted on the independent development set, not on the OOF meta-training set. To avoid fitting the detector on examples used to optimize it, the OOF pool is split into detector training and detector validation subsets. All transformations live inside model pipelines.

In [ ]:
def preprocess(X):
    num=list(X.select_dtypes(include='number').columns)
    cat=[c for c in X.columns if c not in num]
    return ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),
      ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore',sparse_output=False,min_frequency=15))]),cat)],sparse_threshold=0)

def models(X):
    ntrees=35 if FAST_MODE else 85
    return {
      'Logistic Regression':Pipeline([('prep',preprocess(X)),('clf',LogisticRegression(max_iter=450,random_state=SEED))]),
      'Random Forest':Pipeline([('prep',preprocess(X)),('clf',RandomForestClassifier(n_estimators=ntrees,min_samples_leaf=2,n_jobs=-1,random_state=SEED))]),
      'Gradient Boosting':Pipeline([('prep',preprocess(X)),('clf',HistGradientBoostingClassifier(max_iter=55 if FAST_MODE else 100,max_leaf_nodes=15,random_state=SEED))]),
      'MLP':Pipeline([('prep',preprocess(X)),('clf',MLPClassifier(hidden_layer_sizes=(32,),max_iter=110 if FAST_MODE else 170,early_stopping=True,random_state=SEED))])}

def evaluate(y,p):
    p=np.asarray(p); assert p.ndim==2 and p.shape[1]==2 and np.allclose(p.sum(axis=1),1,atol=1e-5)
    yh=p.argmax(axis=1)
    return dict(accuracy=accuracy_score(y,yh),balanced_accuracy=balanced_accuracy_score(y,yh),precision=precision_score(y,yh,zero_division=0),recall=recall_score(y,yh,zero_division=0),f1=f1_score(y,yh,zero_division=0),roc_auc=roc_auc_score(y,p[:,1]),log_loss=log_loss(y,p,labels=[0,1]),brier=brier_score_loss(y,p[:,1]),tn=confusion_matrix(y,yh,labels=[0,1])[0,0],fp=confusion_matrix(y,yh,labels=[0,1])[0,1],fn=confusion_matrix(y,yh,labels=[0,1])[1,0],tp=confusion_matrix(y,yh,labels=[0,1])[1,1])


## 8. Base classifier training
Models use fixed configurations, with candidate selection on development log loss. Cross-fitted base predictions are produced only for the chosen candidate, after development selection. The test set remains unseen.

## 9. Base-model performance evaluation
Report classification metrics on test **only after choices are frozen**. Accuracy can hide error asymmetry, which is why balanced accuracy, precision, recall, F1, AUROC, Brier, and log loss are also reported.

## 10. Prediction error analysis
Inspect confidence by correctness and the confusion matrix. Association with input attributes does not demonstrate causation.

## 11. Maximum confidence baseline
The baseline failure-risk score is `1-max_k P(y=k|x)`. This score exists at inference time.

## 12. Predictive entropy and probability margin
Normalized entropy and inverse top-two margin are computed from the same probability vector. They are not independent signals for a binary classifier.

In [ ]:
def risk_scores(p):
    p=np.clip(np.asarray(p),1e-12,1)
    s=np.sort(p,axis=1)
    return pd.DataFrame({'confidence':1-s[:,-1], 'entropy':-(p*np.log(p)).sum(axis=1)/np.log(p.shape[1]), 'inverse_margin':1-(s[:,-1]-s[:,-2])})

def ece(y,p,bins=12):
    yh=p.argmax(axis=1); conf=p.max(axis=1); correct=(yh==np.asarray(y)).astype(int)
    ix=np.minimum((conf*bins).astype(int),bins-1)
    return sum(np.mean(ix==k)*abs(correct[ix==k].mean()-conf[ix==k].mean()) for k in range(bins) if (ix==k).any())

def reliability(y,p,title,bins=10):
    yh=p.argmax(axis=1);conf=p.max(axis=1);corr=(yh==np.asarray(y)).astype(float)
    edges=np.linspace(0,1,bins+1);xx=[];yy=[];ww=[]
    for lo,hi in zip(edges[:-1],edges[1:]):
        mask=(conf>=lo)&(conf<=hi if hi==1 else conf<hi)
        if mask.sum():xx.append(conf[mask].mean());yy.append(corr[mask].mean());ww.append(mask.sum())
    plt.plot([0,1],[0,1],'k--',lw=1,label='ideal');plt.plot(xx,yy,'o-',label=title)
    plt.xlabel('Mean predicted confidence');plt.ylabel('Fraction correct');plt.xlim(0,1);plt.ylim(0,1);plt.legend();plt.title('Reliability: '+title)

def detection(yerror,risk):
    err=np.asarray(yerror).astype(int);r=np.asarray(risk,dtype=float)
    if err.min()==err.max():return {'error_auroc':np.nan,'error_auprc':np.nan}
    return {'error_auroc':roc_auc_score(err,r),'error_auprc':average_precision_score(err,r)}

def policy(yerror,risk,budgets=(.05,.1,.2,.3)):
    err=np.asarray(yerror).astype(int);risk=np.asarray(risk)
    order=np.argsort(-risk,kind='stable');out=[];n=len(err)
    for budget in budgets:
        review=np.zeros(n,dtype=bool);review[order[:int(np.ceil(n*budget))]]=True
        accepted=~review
        out.append({'review_budget':budget,'coverage':accepted.mean(),'selective_risk':err[accepted].mean() if accepted.any() else np.nan,
           'errors_caught':int(err[review].sum()),'error_recall':err[review].sum()/max(1,err.sum()),
           'rejected_correct':int((1-err[review]).sum()),'missed_errors':int(err[accepted].sum()),
           'review_precision':err[review].mean() if review.any() else np.nan,'false_alert_rate':(1-err[review]).sum()/max(1,(1-err).sum())})
    return pd.DataFrame(out)

def riskcoverage(yerror,risk):
    err=np.asarray(yerror);order=np.argsort(risk,kind='stable');c=np.arange(1,len(err)+1)/len(err)
    r=np.cumsum(err[order])/np.arange(1,len(err)+1)
    return c,r,float(np.trapezoid(r,c))


## 13. Probability calibration evaluation
Reliability diagrams and confidence-ECE summarize correctness calibration, while class log loss/Brier summarize predictive probability quality. ECE depends on chosen bins. Confidence-ECE is not identical to class-probability calibration.

## 14. Post-hoc calibration experiments
Platt/logistic and isotonic mapping plus temperature scaling use a *frozen* fitted base model and development predictions only. For binary temperature scaling, divide logits of base probabilities by a positive temperature. Calibrator comparison uses development log loss. Isotonic and Platt can change ties/rankings in different ways.

In [ ]:
from scipy.optimize import minimize_scalar
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression as LR

def fit_calibrators(ydev,pdev):
    ydev=np.asarray(ydev); raw=np.clip(pdev[:,1],1e-9,1-1e-9)
    logits=np.log(raw/(1-raw))
    lr=LR(max_iter=300).fit(logits.reshape(-1,1),ydev)
    iso=IsotonicRegression(out_of_bounds='clip').fit(raw,ydev)
    def tprob(logit,T):
        z=np.clip(logit/T,-40,40);return 1/(1+np.exp(-z))
    opt=minimize_scalar(lambda t: log_loss(ydev,tprob(logits,np.exp(t)),labels=[0,1]),bounds=(-2.3,2.3),method='bounded')
    T=np.exp(opt.x)
    def transform(p,method):
        z=np.clip(p[:,1],1e-9,1-1e-9);l=np.log(z/(1-z))
        v={'raw':lambda:z,'platt':lambda:lr.predict_proba(l.reshape(-1,1))[:,1],
           'isotonic':lambda:iso.predict(z),'temperature':lambda:tprob(l,T)}[method]()
        v=np.clip(v,1e-8,1-1e-8);return np.column_stack((1-v,v))
    return transform,T


## 15. Ensemble-based uncertainty
A bootstrap ensemble of independently fitted random forests gives predictive entropy, mean member entropy, variance of positive-class probabilities and a nonnegative entropy-gap (mutual-information-style) proxy. An ensemble of correlated forests is not a Bayesian posterior, and the gap is only a heuristic.

In [ ]:
def make_ensemble(Xtrain,ytrain,Xpredict,nmembers=3):
    ps=[];tic=time.perf_counter()
    rng=np.random.default_rng(SEED+323)
    for i in range(nmembers):
        inds=rng.choice(len(ytrain),len(ytrain),replace=True)
        est=Pipeline([('prep',preprocess(Xtrain)),('rf',RandomForestClassifier(n_estimators=25 if FAST_MODE else 45,min_samples_leaf=2,random_state=SEED+100+i,n_jobs=-1))])
        est.fit(Xtrain.iloc[inds],ytrain.iloc[inds]);ps.append(est.predict_proba(Xpredict))
    arr=np.array(ps);mean=arr.mean(axis=0)
    H=lambda z:-(np.clip(z,1e-12,1)*np.log(np.clip(z,1e-12,1))).sum(axis=-1)/np.log(2)
    return pd.DataFrame({'ensemble_entropy':H(mean),'ensemble_variance':arr[:,:,1].var(axis=0),'ensemble_mi':np.maximum(0,H(mean)-H(arr).mean(axis=0)),
       'ensemble_vote_disagreement':(arr.argmax(axis=-1)!=np.round(arr.argmax(axis=-1).mean(axis=0))[None,:]).mean(axis=0)}),time.perf_counter()-tic


## 16. Construction of the error-prediction dataset
Create 3-fold out-of-fold predictions *within the base-training partition*. Each meta training label compares an OOF prediction to the known training label. No observation trains the classifier making its own OOF prediction. Ensemble train-side meta-features are also derived out-of-fold where enabled. We explicitly prevent using in-sample ensemble features in meta training.

## 17. Training dedicated failure-prediction models
Train three detectors (logistic, forest, gradient boosting) on cross-fitted error labels. Stratify an internal detector train/validation split. Select the detector on this detector-validation set. The independent development partition is then used to select calibrators, policies and a final method, leaving the test untouched.

In [ ]:
def meta_features(p,ens=None, X=None,with_features=False):
    z=risk_scores(p)
    z['predicted_class']=np.asarray(p).argmax(axis=1)
    if ens is not None:
        for c in ens:z[c]=np.asarray(ens[c])
    if with_features and X is not None:
        # Numeric raw inputs only; imputed, with explicit missingness indicators; no ground-truth columns
        sub=X.select_dtypes(include='number').iloc[:,:8].reset_index(drop=True)
        for c in sub: z['raw_'+str(c)]=sub[c].fillna(sub[c].median() if sub[c].notna().any() else 0).to_numpy()
    return z.fillna(0)

def detector_models():
    return {'Meta Logistic':Pipeline([('scale',StandardScaler()),('clf',LogisticRegression(class_weight='balanced',max_iter=500,random_state=SEED))]),
      'Meta Forest':RandomForestClassifier(n_estimators=55 if FAST_MODE else 100,min_samples_leaf=4,class_weight='balanced_subsample',random_state=SEED,n_jobs=-1),
      'Meta Boosting':HistGradientBoostingClassifier(max_iter=55 if FAST_MODE else 85,max_leaf_nodes=8,random_state=SEED)}

def fit_meta(Xmeta,yerror):
    if len(np.unique(yerror))<2:raise ValueError('Insufficient OOF errors to fit a failure detector')
    ii,jj=train_test_split(np.arange(len(yerror)),test_size=.3,stratify=yerror,random_state=SEED+23)
    scores=[];fitted={}
    for name,est in detector_models().items():
        tic=time.perf_counter();est.fit(Xmeta.iloc[ii],np.asarray(yerror)[ii]);q=est.predict_proba(Xmeta.iloc[jj])[:,1]
        scores.append({'detector':name,**detection(np.asarray(yerror)[jj],q),'fit_seconds':time.perf_counter()-tic})
    ranks=pd.DataFrame(scores).sort_values(['error_auprc','error_auroc'],ascending=False)
    winner=ranks.iloc[0]['detector'];final=detector_models()[winner].fit(Xmeta,np.asarray(yerror))
    return final,ranks


## 18. Failure-detection evaluation and leaderboard
Error AUROC and AUPRC use errors as positive. AUPRC must be judged against observed error prevalence, which changes across models and shifts. Show ROC and precision–recall curves, detection precision/recall at review budgets, false alerts, and inference overhead.

## 19. High-confidence failure analysis
Count confidence ≥0.90 and ≥0.95 errors, show their predicted classes and attributes, and quantify those flagged by a budget-limited learned detector. High-confidence error patterns do not establish causes.

## 20. Selective prediction and abstention
Human-review referrals are ranked by risk. Report referrals that were actually correct alongside errors retained without review. A purported safety policy must report both.

## 21. Risk–coverage and review-budget analysis
For a risk score, sort predictions from lowest to highest risk. At $k/n$ coverage, selective risk is the error fraction among the $k$ retained predictions. **AURC** is trapezoidal integration of the risk–coverage curve on the observed grid (near-zero coverage excluded). Budget policies refer the highest-risk 5%, 10%, 20%, and 30%. Budgets are preset, not chosen on test.

## 22. Distribution shift and robustness
Evaluate additive measurement noise, MCAR additional numeric missingness, a numeric drift shift, and a predefined demographic/subgroup slice when available. We *hold labels fixed* as a sensitivity analysis, not a claim that every perturbed record still represents a valid real-world case. The original trained pipeline handles missing features. Detector generalization is assessed without refitting on corrupted test data.

In [ ]:
def corrupt(X,kind,rng):
    z=X.copy()
    nums=list(z.select_dtypes(include='number').columns)
    if kind=='noise' and nums:
        for c in nums:
            sd=z[c].std();z[c]=z[c]+rng.normal(0,.35*(sd if np.isfinite(sd) else 0),len(z))
    elif kind=='missingness' and nums:
        for c in nums:
            mask=rng.random(len(z))<.12;z.loc[mask,c]=np.nan
    elif kind=='drift' and nums:
        for c in nums[:max(1,len(nums)//3)]:
            sd=z[c].std();z[c]=z[c]+.65*(sd if np.isfinite(sd) else 0)
    return z

def bootstrap_difference(err,a,b,iterations=250):
    rng=np.random.default_rng(SEED+29);err=np.asarray(err);a=np.asarray(a);b=np.asarray(b)
    if err.min()==err.max():return (np.nan,np.nan,np.nan)
    dif=[]
    for _ in range(iterations):
        ix=rng.integers(0,len(err),len(err));yy=err[ix]
        if len(np.unique(yy))==2:dif.append(roc_auc_score(yy,a[ix])-roc_auc_score(yy,b[ix]))
    if not dif:return (np.nan,np.nan,np.nan)
    return (np.mean(dif),*np.quantile(dif,[.025,.975]))


## 23. Statistical significance and ablation studies
Bootstrap a paired difference in detector error-AUROC on common test examples and show a percentile 95% interval. It is exploratory (limited resamples, no correction for repeated comparisons). Ablate confidence alone, confidence+entropy+margin, plus predicted class, plus raw numerical input features, and plus ensemble disagreement. All detector ablations are trained on the OOF predictions. OOF ensemble features are expensive and disabled in FAST_MODE. The final test is used for estimation only, never for choosing the winning ablation.

## Experiment runner (sections 8–23)
The unified runner ensures test calls happen only after model, calibration, and detector choices are completed. All prediction-level and summary data are retained for reproducibility.

In [ ]:
ALL_BASE=[];ALL_CAL=[];ALL_DET=[];ALL_POLICY=[];ALL_SHIFT=[];ALL_PREDS=[];ALL_ABL=[];ALL_HIGH=[];ALL_TIMES=[]
for dataset in RUN_DATASETS:
    print('\n'+'='*75+'\nDATASET:',dataset)
    X,y,source=get_dataset(dataset)
    X=X.reset_index(drop=True);y=y.reset_index(drop=True)
    if FAST_MODE and dataset=='adult':
        X,_,y,_=train_test_split(X,y,train_size=min(9000,len(y)-1),stratify=y,random_state=SEED)
        X=X.reset_index(drop=True);y=y.reset_index(drop=True)
    print('Source:',source,'rows',len(y),'class counts',y.value_counts().to_dict())
    eda(X,y,dataset)
    Xa,Xd,Xt,ya,yd,yt=make_split(X,y)
    print('Partition sizes: train, dev, test',len(ya),len(yd),len(yt))
    factory=models(Xa)
    fitted={};devscores=[];fit_times={}
    for name,model in factory.items():
        tic=time.perf_counter();model.fit(Xa,ya);secs=time.perf_counter()-tic
        pdv=model.predict_proba(Xd);devscores.append({'model':name,'dev_logloss':log_loss(yd,pdv,labels=[0,1]),'dev_auroc':roc_auc_score(yd,pdv[:,1])})
        fitted[name]=model;fit_times[name]=secs
        print(name, 'development log loss',round(devscores[-1]['dev_logloss'],4),'fit seconds',round(secs,2))
    devtable=pd.DataFrame(devscores).sort_values('dev_logloss');display(devtable)
    selected=devtable.iloc[0]['model'];base=fitted[selected]
    print('FROZEN base selection:',selected)
    # Cross-fitted OOF generation (a prediction never comes from an estimator fit on its own row)
    skf=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED)
    oof=cross_val_predict(factory[selected],Xa,ya,cv=skf,method='predict_proba',n_jobs=1)
    assert oof.shape==(len(ya),2) and np.allclose(oof.sum(axis=1),1)
    error_oof=(oof.argmax(axis=1)!=ya.to_numpy()).astype(int)
    print('OOF errors',error_oof.sum(),'rate',round(error_oof.mean(),4))
    # Ensemble features for detector training are cross-fitted too, not in-sample
    ens_oof=None
    if not FAST_MODE:
        ens_oof=pd.DataFrame(index=np.arange(len(ya)),columns=['ensemble_entropy','ensemble_variance','ensemble_mi','ensemble_vote_disagreement'],dtype=float)
        for fit_idx,val_idx in skf.split(Xa,ya):
            f,_=make_ensemble(Xa.iloc[fit_idx],ya.iloc[fit_idx],Xa.iloc[val_idx],nmembers=3)
            ens_oof.iloc[val_idx]=f.to_numpy()
        assert ens_oof.notna().all().all()
    # These are deployment-time features from development data, generated by held-out base fit
    pdev=base.predict_proba(Xd)
    ens_dev,ens_dev_seconds=make_ensemble(Xa,ya,Xd,nmembers=3)
    configs={'confidence':['confidence'], 'uncertainty_triple':['confidence','entropy','inverse_margin'],
      'plus_predicted_class':['confidence','entropy','inverse_margin','predicted_class'],
      'plus_numeric_inputs':None}
    meta_oof=meta_features(oof,ens_oof,Xa,with_features=True)
    meta_dev=meta_features(pdev,ens_dev,Xd,with_features=True)
    # For ensemble ablations meta train features must have OOF analogue
    if ens_oof is not None:configs['plus_ensemble']=['confidence','entropy','inverse_margin','predicted_class']+list(ens_oof.columns)
    configs['plus_numeric_inputs']=list(meta_oof.columns[:4])+[c for c in meta_oof.columns if c.startswith('raw_')]
    meta_scores=[];meta_fits={}
    for ab,cols in configs.items():
        detector,validation=fit_meta(meta_oof[cols],error_oof)
        meta_fits[ab]=(detector,cols)
        dscore=detector.predict_proba(meta_dev[cols])[:,1]
        meta_scores.append({'dataset':dataset,'ablation':ab,'detector':validation.iloc[0]['detector'],
                            **detection((pdev.argmax(axis=1)!=yd.to_numpy()).astype(int),dscore),
                            'dev_ap':detection((pdev.argmax(axis=1)!=yd.to_numpy()).astype(int),dscore)['error_auprc']})
    mscore=pd.DataFrame(meta_scores).sort_values(['dev_ap','ablation'],ascending=[False,True]);display(mscore)
    picked=mscore.iloc[0]['ablation'];meta,meta_cols=meta_fits[picked]
    print('FROZEN detector:',picked,'+ internal detector model')
    # Fit calibrated mapping using dev and select ONLY on development logloss
    transform,temp=fit_calibrators(yd,pdev)
    for method in ['raw','platt','isotonic','temperature']:
        q=transform(pdev,method); ALL_CAL.append({'dataset':dataset,'split':'development','method':method,'log_loss':log_loss(yd,q,labels=[0,1]),'brier':brier_score_loss(yd,q[:,1]),'ece_12':ece(yd,q,12)})
    cal_choice=min((r for r in ALL_CAL if r['dataset']==dataset and r['split']=='development'),key=lambda r:r['log_loss'])['method']
    print('FROZEN calibration method:',cal_choice,'temperature=',round(temp,3))
    # Freeze selection among candidate risk methods using development failure average precision
    err_dev=(pdev.argmax(axis=1)!=yd.to_numpy()).astype(int)
    risk_dev=risk_scores(pdev)
    risk_dev['ensemble_mi']=ens_dev['ensemble_mi'].to_numpy()
    risk_dev['learned_error']=meta.predict_proba(meta_dev[meta_cols])[:,1]
    method_scores={m:detection(err_dev,risk_dev[m])['error_auprc'] for m in risk_dev}
    policy_choice=max(method_scores,key=method_scores.get)
    print('FROZEN deployment risk method:',policy_choice,method_scores)
    # FINAL TEST BEGINS. No selections or refitting after here.
    for name,model in fitted.items():
        tic=time.perf_counter();pt=model.predict_proba(Xt);inference=time.perf_counter()-tic
        ALL_BASE.append({'dataset':dataset,'model':name,'selected':name==selected,'fit_seconds':fit_times[name], 'inference_seconds':inference,**evaluate(yt,pt)})
        yh=pt.argmax(axis=1);hc=((yh!=yt.to_numpy()) & (pt.max(axis=1)>=.9))
        ALL_HIGH.append({'dataset':dataset,'model':name,'threshold':.90,'high_conf_errors':int(hc.sum()),'all_errors':int((yh!=yt.to_numpy()).sum())})
        hc95=((yh!=yt.to_numpy()) & (pt.max(axis=1)>=.95))
        ALL_HIGH.append({'dataset':dataset,'model':name,'threshold':.95,'high_conf_errors':int(hc95.sum()),'all_errors':int((yh!=yt.to_numpy()).sum())})
    tic=time.perf_counter();ptest=base.predict_proba(Xt);base_inf=time.perf_counter()-tic
    ens_test,ens_test_seconds=make_ensemble(Xa,ya,Xt,nmembers=3)
    meta_test=meta_features(ptest,ens_test,Xt,with_features=True)
    tic=time.perf_counter();meta_test_score=meta.predict_proba(meta_test[meta_cols])[:,1];meta_seconds=time.perf_counter()-tic
    err=(ptest.argmax(axis=1)!=yt.to_numpy()).astype(int)
    test_scores=risk_scores(ptest);test_scores['ensemble_entropy']=ens_test['ensemble_entropy'].to_numpy();test_scores['ensemble_variance']=ens_test['ensemble_variance'].to_numpy();test_scores['ensemble_mi']=ens_test['ensemble_mi'].to_numpy()
    test_scores['learned_error']=meta_test_score
    caltest=transform(ptest,cal_choice)
    for method in ['raw','platt','isotonic','temperature']:
        q=transform(ptest,method); ALL_CAL.append({'dataset':dataset,'split':'test','method':method,'log_loss':log_loss(yt,q,labels=[0,1]),'brier':brier_score_loss(yt,q[:,1]),'ece_12':ece(yt,q,12)})
    for method in test_scores:
        r=test_scores[method].to_numpy();met=detection(err,r)
        ALL_DET.append({'dataset':dataset,'selected_base':selected,'detector':method,'chosen_policy':method==policy_choice,
                        'error_prevalence':err.mean(),**met,'aurc':riskcoverage(err,r)[2]})
        tmp=policy(err,r);tmp['dataset']=dataset;tmp['method']=method;ALL_POLICY.append(tmp)
    # Ablations are comparative TEST reporting, not selection
    for ab,(mdl,cols) in meta_fits.items():
        sc=mdl.predict_proba(meta_test[cols])[:,1]
        ALL_ABL.append({'dataset':dataset,'ablation':ab,'selected':ab==picked,**detection(err,sc)})
    predout=pd.DataFrame({'dataset':dataset,'row_id':Xt.index,'true_class':yt.to_numpy(),'predicted_class':ptest.argmax(axis=1),
                          'prob_class_1':ptest[:,1],'confidence':ptest.max(axis=1),'error':err,'calibrated_prob_class_1':caltest[:,1]})
    for k in test_scores:predout['risk_'+k]=test_scores[k].to_numpy()
    ALL_PREDS.append(predout)
    print('Classification metrics:');display(pd.DataFrame([r for r in ALL_BASE if r['dataset']==dataset]).round(3))
    print('Failure leaderboard:');display(pd.DataFrame([r for r in ALL_DET if r['dataset']==dataset]).sort_values('error_auprc',ascending=False).round(3))
    # Figures 1-14 on measured predictions
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    bb=pd.DataFrame([r for r in ALL_BASE if r['dataset']==dataset]);ax[0].barh(bb['model'],bb['accuracy']);ax[0].set_xlim(0,1);ax[0].set_title('Test accuracy | '+dataset)
    ax[1].hist(ptest.max(axis=1)[err==0],bins=16,alpha=.6,density=True,label='correct');ax[1].hist(ptest.max(axis=1)[err==1],bins=16,alpha=.6,density=True,label='wrong');ax[1].legend();ax[1].set_title('Maximum confidence by correctness');plt.tight_layout();plt.show()
    fig,ax=plt.subplots(1,2,figsize=(12,4));plt.sca(ax[0]);reliability(yt,ptest,'uncalibrated');plt.sca(ax[1]);reliability(yt,caltest,'calibrated: '+cal_choice);plt.tight_layout();plt.show()
    fig,ax=plt.subplots(1,3,figsize=(15,4))
    for method in ['confidence','ensemble_mi','learned_error']:
        if len(np.unique(err))==2:
            fpr,tpr,_=roc_curve(err,test_scores[method]);ax[0].plot(fpr,tpr,label=method)
            precision,recall,_=precision_recall_curve(err,test_scores[method]);ax[1].plot(recall,precision,label=method)
        coverage,risk,aurc=riskcoverage(err,test_scores[method]);ax[2].plot(coverage,risk,label=f'{method}, AURC {aurc:.3f}')
    ax[0].set(xlabel='False positive rate',ylabel='Error recall',title='Error detection ROC');ax[1].set(xlabel='Error recall',ylabel='Precision',title='Error detection PR');ax[2].set(xlabel='Automatic coverage',ylabel='Accepted-case error rate',title='Risk–coverage')
    for a in ax:a.legend(fontsize=8)
    plt.tight_layout();plt.show()
    fig,ax=plt.subplots(1,3,figsize=(15,4))
    for method in ['confidence','ensemble_mi','learned_error']:
        bud=policy(err,test_scores[method]);ax[0].plot(bud.review_budget,bud.error_recall,'o-',label=method)
    ax[0].legend();ax[0].set(xlabel='Review budget',ylabel='Fraction of errors caught',title='Review-budget effectiveness')
    ax[1].hist(test_scores['entropy'][err==0],alpha=.6,bins=15,label='correct');ax[1].hist(test_scores['entropy'][err==1],alpha=.6,bins=15,label='wrong');ax[1].legend();ax[1].set_title('Entropy by correctness')
    ax[2].hist(test_scores['ensemble_mi'][err==0],alpha=.6,bins=15,label='correct');ax[2].hist(test_scores['ensemble_mi'][err==1],alpha=.6,bins=15,label='wrong');ax[2].legend();ax[2].set_title('Ensemble disagreement by correctness');plt.tight_layout();plt.show()
    hc=predout[(predout.error==1)&(predout.confidence>=.9)].sort_values('confidence',ascending=False)
    print('High-confidence wrong >=.90:',len(hc),'>=.95:',(hc.confidence>=.95).sum())
    display(hc.head(12));print('Predicted classes of high-confidence errors:',hc.predicted_class.value_counts().to_dict())
    print('High-confidence errors flagged at 20% review budget:',int((hc.row_id.isin(predout.iloc[np.argsort(-test_scores[policy_choice].to_numpy())[:int(np.ceil(len(err)*.2))]].row_id)).sum()))
    # controlled shifts
    for shift in ['original','noise','missingness','drift']:
        xx=Xt if shift=='original' else corrupt(Xt,shift,np.random.default_rng(SEED+{'noise':1,'missingness':2,'drift':3}[shift]))
        p=base.predict_proba(xx);e=(p.argmax(axis=1)!=yt.to_numpy()).astype(int)
        ens,_=make_ensemble(Xa,ya,xx,nmembers=3)
        mm=meta_features(p,ens,xx,with_features=True)
        sr=meta.predict_proba(mm[meta_cols])[:,1]
        ALL_SHIFT.append({'dataset':dataset,'shift':shift,'accuracy':accuracy_score(yt,p.argmax(axis=1)),
          'ece_12':ece(yt,p),'mean_confidence':p.max(axis=1).mean(),'error_count':e.sum(),
          **detection(e,sr),'selective_risk_80pct':policy(e,sr,budgets=(.2,)).iloc[0]['selective_risk']})
    ss=pd.DataFrame([r for r in ALL_SHIFT if r['dataset']==dataset]);display(ss.round(3))
    ss.plot(x='shift',y=['accuracy','error_auroc','selective_risk_80pct'],kind='bar',title='Clean vs corrupted data: '+dataset,rot=0);plt.ylim(0,1);plt.tight_layout();plt.show()
    ab=pd.DataFrame([r for r in ALL_ABL if r['dataset']==dataset]);ab.plot.bar(x='ablation',y='error_auprc',legend=False,title='Learned detector ablations: '+dataset);plt.xticks(rotation=35,ha='right');plt.tight_layout();plt.show()
    dlt=bootstrap_difference(err,test_scores['learned_error'],test_scores['confidence'],iterations=100 if FAST_MODE else 350)
    print('Paired bootstrap delta error-AUROC (learned - confidence), mean and 95% CI:',dlt)
    print('Rank correlation confidence/entropy:',pd.Series(test_scores['confidence']).corr(pd.Series(test_scores['entropy']),method='spearman'))
    ALL_TIMES.append({'dataset':dataset,'base_fit_seconds':fit_times[selected], 'base_test_inference_seconds':base_inf,
                      'ensemble_test_seconds':ens_test_seconds,'meta_test_seconds':meta_seconds,'error_prevalence':err.mean(),
                      'base_model':selected,'detector_ablation':picked,'calibration_choice':cal_choice,'policy_choice':policy_choice,
                      'bootstrap_auc_delta_mean':dlt[0],'bootstrap_auc_delta_ci_lower':dlt[1],'bootstrap_auc_delta_ci_upper':dlt[2]})


## 24. Consolidated results and research discussion
The following scorecards are built from measured predictions. **Interpretation guide:** if a learned detector's paired bootstrap CI crosses zero, avoid concluding it is better than confidence. If a method has low PR compared with error prevalence, it misses many errors even if ROC-AUC looks plausible. ECE should be contrasted with Brier and log loss. Shift conditions are artificial sensitivity probes rather than validated natural OOD data.

**Answers to the eight questions** are generated from observed tables below. The notebook intentionally does not present invented aggregate statistics.

In [ ]:
base_df=pd.DataFrame(ALL_BASE);cal_df=pd.DataFrame(ALL_CAL);det_df=pd.DataFrame(ALL_DET)
policy_df=pd.concat(ALL_POLICY,ignore_index=True);shift_df=pd.DataFrame(ALL_SHIFT)
pred_df=pd.concat(ALL_PREDS,ignore_index=True);ab_df=pd.DataFrame(ALL_ABL);time_df=pd.DataFrame(ALL_TIMES)
exports={'prediction_level_results.csv':pred_df,'base_model_performance.csv':base_df,'calibration_results.csv':cal_df,
 'failure_detection_leaderboard.csv':det_df,'risk_coverage_results.csv':policy_df,
 'distribution_shift_comparison.csv':shift_df,'meta_ablation_results.csv':ab_df,
 'runtime_and_selection.csv':time_df,'high_confidence_by_model.csv':pd.DataFrame(ALL_HIGH)}
for filename,frame in exports.items():frame.to_csv(OUT/filename,index=False);print('saved',OUT/filename,frame.shape)
display(time_df.round(4));display(det_df.sort_values(['dataset','error_auprc'],ascending=[True,False]).round(4))
for dataset in time_df.dataset:
    row=time_df[time_df.dataset==dataset].iloc[0]
    d=det_df[det_df.dataset==dataset].set_index('detector')
    b=base_df[(base_df.dataset==dataset)&(base_df.selected)].iloc[0]
    c=cal_df[(cal_df.dataset==dataset)&(cal_df.split=='test')].set_index('method')
    po=policy_df[(policy_df.dataset==dataset)&(policy_df.method==row.policy_choice)&(np.isclose(policy_df.review_budget,.2))].iloc[0]
    hi=pred_df[(pred_df.dataset==dataset)&(pred_df.error==1)&(pred_df.confidence>=.9)]
    print('\nRESEARCH FINDINGS:',dataset)
    print(f'Q1: {len(hi)} wrong predictions had >=90% confidence among {int((pred_df.dataset==dataset).sum())} test cases.')
    print(f'Q2–3: confidence error-AUROC {d.loc["confidence","error_auroc"]:.3f}, ensemble MI {d.loc["ensemble_mi","error_auroc"]:.3f}. Binary entropy and margin rankings are equivalent to confidence.')
    print(f'Q4: selected {row.calibration_choice}; raw test logloss={c.loc["raw","log_loss"]:.3f}; calibrated test logloss={c.loc[row.calibration_choice,"log_loss"]:.3f}. Selection was based on development logloss.')
    print(f'Q5: learned error AUROC={d.loc["learned_error","error_auroc"]:.3f} vs confidence={d.loc["confidence","error_auroc"]:.3f}. Paired CI for difference [{row.bootstrap_auc_delta_ci_lower:.3f}, {row.bootstrap_auc_delta_ci_upper:.3f}].')
    print(f'Q6: using {row.policy_choice}, at 20% review budget retained error rate={po.selective_risk:.3f}, raw error rate={1-b.accuracy:.3f}; missed errors={int(po.missed_errors)}, unnecessary referrals={int(po.rejected_correct)}.')
    ss=shift_df[shift_df.dataset==dataset].set_index('shift')
    print('Q7: clean vs noise accuracy',round(ss.loc['original','accuracy'],3),'/',round(ss.loc['noise','accuracy'],3),'and detector error AUROC',round(ss.loc['original','error_auroc'],3),'/',round(ss.loc['noise','error_auroc'],3))
    print(f'Q8: base fit {row.base_fit_seconds:.2f}s; ensemble test inference {row.ensemble_test_seconds:.2f}s; learned detector test inference {row.meta_test_seconds:.3f}s. Weigh review quality and cost, not only scores.')
with open(OUT/'reproducibility.json','w') as f:
    json.dump({'seed':SEED,'fast_mode':FAST_MODE,'datasets':RUN_DATASETS,'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__,
      'sklearn':sklearn.__version__,'sources':['UCI WDBC dataset 17','UCI Adult dataset 2 / OpenML 1590'],
      'split':'stratified 60% base-training, 20% development, 20% test; 3-fold OOF meta training'},f,indent=2)


## 25. Limitations, conclusions, references, and future work

**Interpret results above rather than assuming an outcome.** A positive error-AUROC does not make an infallible rejection system. Precision–recall is baseline-dependent. Detector training needs historical labels and an adequate number of cross-fitted errors. A selected detector can overfit the independent development set through repeated comparisons. Observational subgroup differences are not causal claims. No dataset here demonstrates clinical suitability or equitable deployment. Adult is historical U.S. census data and WDBC is a small clinical benchmark. Add repeated seeds, paired outer resampling, true external shift, decision-theoretic cost evaluation, subgroup-specific calibration, a broader dataset collection, and prospective monitoring before any deployment claim.

**Important design caveats:** the calibration dataset is also used for model selection and policy comparison. This remains test-disjoint, but a more conservative deployment benchmark would use additional disjoint calibration and policy-selection partitions. The OOF meta-target is cross-fitted, though it may not perfectly match the distribution of predictions from a model fit on the entire training pool. Human review is treated as perfect only for counting referred errors, not validated reviewer outcomes. Fixed-label numeric corruptions need not preserve real-world data semantics.

**References**
1. Guo et al. (2017), *On Calibration of Modern Neural Networks*, ICML, https://arxiv.org/abs/1706.04599
2. Lakshminarayanan et al. (2017), *Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles*, NeurIPS, https://arxiv.org/abs/1612.01474
3. Geifman and El-Yaniv (2017), *Selective Classification for Deep Neural Networks*, NeurIPS, https://arxiv.org/abs/1705.08500
4. Hendrycks and Gimpel (2017), *A Baseline for Detecting Misclassified and Out-of-Distribution Examples in Neural Networks*, ICLR, https://arxiv.org/abs/1610.02136
5. Naeini, Cooper and Hauskrecht (2015), *Obtaining Well Calibrated Probabilities Using Bayesian Binning*, AAAI, https://doi.org/10.1609/aaai.v29i1.9602
6. [UCI WDBC](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic), [UCI Adult](https://archive.ics.uci.edu/dataset/2/adult), [OpenML Adult](https://www.openml.org/d/1590).

**Reproduction:** rerun every cell from the top on Kaggle. Output files appear in the notebook's Output pane from `/kaggle/working/`. For full research results leave `FAST_MODE=0`. Any optional input must match the original UCI schema.

## Optional: download experiment artifacts in Colab

After the study completes, run the cell below to create a ZIP of the output CSV files. On Google Colab, set `DOWNLOAD_RESULTS=True` to initiate a browser download. If downloads are blocked, retrieve the ZIP from the Colab Files sidebar instead.

In [ ]:
from pathlib import Path
import zipfile

DOWNLOAD_RESULTS = False  # Set True only if you want Colab to download the ZIP automatically.
zip_path = OUT.parent / 'failure_benchmark_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for filename in sorted(OUT.glob('*')):
        if filename.is_file():
            archive.write(filename, arcname=filename.name)
print('Results ZIP:', zip_path)
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except ImportError:
        print('Not in Colab. Download the ZIP directly from:', zip_path)